# 13 - Semi-supervised: use the 5000 unlabeled TEST rows (riskier, higher ceiling)

Everything so far ignored a big asset: the **5000 test rows**. Their *features* are
provided, so using them (not their labels) is allowed by the rules - this is
semi-supervised learning, not external data. Three ideas, lowest-risk first:

1. **Transductive imputation** (low risk) - fit the `eog_burst_index` imputer on
   train+test features (14000 rows) instead of train alone (9000). Strengthens the one
   lever that has ever moved this score, with 56% more data and no label leakage.
2. **Self-training / pseudo-labeling** (THE risky bet) - label confident test rows with
   the SVM and retrain on train+pseudo, so the boundary sees the test manifold. High
   ceiling, but confirmation bias can amplify class-2 errors - so we exclude class 2
   from pseudo-labels and **validate honestly on held-out real labels** before trusting it.
3. **Prior-matching adjustment** (cheap) - nudge predicted class proportions toward the
   known ~balanced prior; macro-F1 punishes any under-predicted class (class 2).

Each idea writes its own submission CSV so you can A/B them on the leaderboard.

## Step 0 - Imports and data

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os, numpy as np, pandas as pd
from sklearn.experimental import enable_iterative_imputer   # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import f1_score, classification_report
from scipy import stats

TRAIN_PATH, TEST_PATH, OUT_DIR = '../train.csv', '../test.csv', '../outputs'
train = pd.read_csv(TRAIN_PATH); test = pd.read_csv(TEST_PATH)
FEATURES = [c for c in train.columns if c not in ('id','sleep_stage')]
y = train['sleep_stage'].to_numpy()
PRIOR = np.bincount(y) / len(y)        # ~[0.25,0.25,0.25,0.25] target proportions
SVM_PARAMS = dict(kernel='rbf', C=12, gamma=0.012, probability=True, random_state=42)
print('train', train.shape, '| test', test.shape, '| class prior', np.round(PRIOR,3))

train (9000, 23) | test (5000, 22) | class prior [0.222 0.271 0.249 0.258]


## Step 1 - Idea 2: transductive imputation (fit the imputer on train+test)

The imputer learns 'predict `eog_burst_index` from the other 20 features'. That
relationship is **unsupervised** (no `sleep_stage` involved), so fitting it on all
14000 feature rows leaks no labels - it just gives the regression more data. We fit it
**once** here and reuse the imputed arrays everywhere below (downstream the only
per-fold steps are scaling + SVM).

We first check transductive imputation doesn't *hurt* vs the per-fold imputer baseline.

In [2]:
# Fit ONE imputer on all available feature rows (train + test), transform both.
Xfull = pd.concat([train[FEATURES], test[FEATURES]], axis=0, ignore_index=True)
imp = IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42)
Xfull_imp = imp.fit_transform(Xfull)
Xtr_imp = Xfull_imp[:len(train)]          # imputed train features (9000, 21)
Xte_imp = Xfull_imp[len(train):]          # imputed test features  (5000, 21)
print('imputed train', Xtr_imp.shape, '| imputed test', Xte_imp.shape)

def svm_on_imputed():
    # downstream of imputation: scale -> RBF SVM. (no imputer here; arrays are filled)
    return make_pipeline(StandardScaler(), SVC(**SVM_PARAMS))

# Sanity CV: transductive-imputed features should be >= the per-fold baseline (~0.842).
cv = StratifiedKFold(5, shuffle=True, random_state=42)
oof_trans = np.asarray(cross_val_predict(svm_on_imputed(), Xtr_imp, y, cv=cv,
                                         method='predict_proba', n_jobs=5))
print('transductive-impute SVM OOF macro-F1 = %.4f  (baseline per-fold impute ~0.842)'
      % f1_score(y, oof_trans.argmax(1), average='macro'))

imputed train (9000, 21) | imputed test (5000, 21)
transductive-impute SVM OOF macro-F1 = 0.8401  (baseline per-fold impute ~0.842)


## Step 2 - Idea 1: self-training functions

`self_train` trains the SVM on the current labeled set, pseudo-labels the unlabeled
**pool** (the test rows), keeps only predictions with `proba >= CONF` whose class is
not in `EXCLUDE` (we exclude the weak class 2 so we never reinforce its mistakes),
adds them as extra training rows, and refits. `ROUNDS` repeats the loop, re-labeling
the full pool each round so confidences can grow.

In [3]:
CONF    = 0.90          # confidence threshold for accepting a pseudo-label
EXCLUDE = {2}           # never pseudo-label the weak/capping class
ROUNDS  = 1             # self-training iterations (1-2 is plenty; more risks drift)

def self_train(X_lab, y_lab, X_pool, conf=CONF, exclude=EXCLUDE, rounds=ROUNDS,
               verbose=False):
    cur_X, cur_y = X_lab, y_lab
    for r in range(rounds):
        mdl = svm_on_imputed().fit(cur_X, cur_y)
        proba = mdl.predict_proba(X_pool)
        pred  = proba.argmax(1); pmax = proba.max(1)
        keep  = (pmax >= conf) & (~np.isin(pred, list(exclude)))
        cur_X = np.vstack([X_lab, X_pool[keep]])
        cur_y = np.concatenate([y_lab, pred[keep]])
        if verbose:
            print('  round %d: kept %d/%d pseudo-labels (%.0f%%)'
                  % (r+1, keep.sum(), len(X_pool), 100*keep.mean()))
    return svm_on_imputed().fit(cur_X, cur_y)

## Step 3 - VALIDATE the risky bet honestly (held-out real labels)

Self-training uses the test pool, so we can't CV it the usual way. Instead we **simulate
reality**: hold out 20% of *train* as a labeled eval set, use the other 80% as the
labeled set, and use the **real test rows as the unlabeled pool** - exactly the live
setup. Then we score the self-trained model against the held-out *real* labels and
compare to the plain SVM. Repeat over seeds + paired t-test.

**Decision rule:** only submit the self-trained model if it beats plain SVM on held-out
real labels by clearly more than the ~+/-0.008 noise. (Slow cell: a few min.)

In [4]:
SEEDS = [42, 2024, 7]
plain_f1, self_f1 = [], []
for seed in SEEDS:
    Xtr, Xev, ytr, yev = train_test_split(Xtr_imp, y, test_size=0.2,
                                          stratify=y, random_state=seed)
    plain = svm_on_imputed().fit(Xtr, ytr)
    selft = self_train(Xtr, ytr, Xte_imp, verbose=True)
    pf = f1_score(yev, plain.predict(Xev), average='macro')
    sf = f1_score(yev, selft.predict(Xev), average='macro')
    plain_f1.append(pf); self_f1.append(sf)
    print('seed %-4d  plain=%.4f  self-train=%.4f  (gain %+.4f)' % (seed, pf, sf, sf-pf), flush=True)

plain_f1, self_f1 = np.array(plain_f1), np.array(self_f1)
g = self_f1 - plain_f1
print('\nmean gain (self-train - plain) = %+.4f +/- %.4f over %d held-out evals'
      % (g.mean(), g.std(), len(g)))
print('self-train wins %d/%d   paired p = %.2e' % ((g>0).sum(), len(g),
      stats.ttest_rel(self_f1, plain_f1).pvalue if len(g)>1 else float('nan')))
print('\n-> Submit self-train ONLY if this gain is clearly > ~0.008 and wins most evals.')

  round 1: kept 2179/5000 pseudo-labels (44%)
seed 42    plain=0.8505  self-train=0.8509  (gain +0.0004)
  round 1: kept 2162/5000 pseudo-labels (43%)
seed 2024  plain=0.8421  self-train=0.8432  (gain +0.0011)
  round 1: kept 2212/5000 pseudo-labels (44%)
seed 7     plain=0.8478  self-train=0.8485  (gain +0.0007)

mean gain (self-train - plain) = +0.0008 +/- 0.0003 over 3 held-out evals
self-train wins 3/3   paired p = 6.17e-02

-> Submit self-train ONLY if this gain is clearly > ~0.008 and wins most evals.


## Step 4 - Idea 3: prior-matching adjustment (validated nested)

macro-F1 punishes any class the model under-predicts. We rescale the probability columns
by a weight vector so the predicted class proportions move toward the known `PRIOR`
(~25% each). We validate it **nested**: fit the weights on held-in OOF rows, score on
held-out rows, so the gain is measured on data the weights never saw.

In [5]:
def match_prior_weights(proba, prior=PRIOR, iters=60, damp=0.5):
    """Iteratively scale class columns so argmax proportions approach `prior`."""
    w = np.ones(proba.shape[1])
    for _ in range(iters):
        pred = (proba*w).argmax(1)
        cur = np.bincount(pred, minlength=len(w)) / len(pred)
        w = w * (prior / np.maximum(cur, 1e-6))**damp
        w = w / w.mean()        # only relative scale matters
    return w

# Nested validation on the transductive OOF probabilities from Step 1.
nb, nt = [], []
for seed in [42, 2024, 7]:
    kf = StratifiedKFold(5, shuffle=True, random_state=seed)
    for fit_idx, ev_idx in kf.split(oof_trans, y):
        w = match_prior_weights(oof_trans[fit_idx])
        nb.append(f1_score(y[ev_idx], oof_trans[ev_idx].argmax(1), average='macro'))
        nt.append(f1_score(y[ev_idx], (oof_trans[ev_idx]*w).argmax(1), average='macro'))
nb, nt = np.array(nb), np.array(nt)
pg = nt - nb
print('prior-match (nested): mean gain = %+.4f +/- %.4f over %d folds' % (pg.mean(), pg.std(), len(pg)))
print('wins %d/%d   paired p = %.2e' % ((pg>0).sum(), len(pg), stats.ttest_rel(nt, nb).pvalue))
PRIOR_W = match_prior_weights(oof_trans)   # weights fit on full OOF for the submission
print('prior weights:', np.round(PRIOR_W,3))

prior-match (nested): mean gain = -0.0001 +/- 0.0014 over 15 folds
wins 6/15   paired p = 7.58e-01
prior weights: [1.038 0.988 0.968 1.005]


## Step 5 - Build submissions (self-contained, one CSV per idea to A/B on the LB)

- `svm_transductive_submission.csv` - idea 2 only (safe upgrade of the 0.855 SVM).
- `svm_selftrain_submission.csv` - ideas 2+1 (the risky bet; submit if Step 3 passed).
- `svm_selftrain_prior_submission.csv` - ideas 2+1+3 (also gated on Step 4).

In [6]:
os.makedirs(OUT_DIR, exist_ok=True)
def write_sub(pred, tag):
    sub = pd.DataFrame({'id': test['id'], 'sleep_stage': pred.astype(int)})
    p = os.path.join(OUT_DIR, f'{tag}_submission.csv'); sub.to_csv(p, index=False)
    print('wrote', p, '| dist', dict(sub.sleep_stage.value_counts().sort_index()))
    return sub

# Idea 2: transductive-impute SVM on test
trans_model = svm_on_imputed().fit(Xtr_imp, y)
proba_trans = trans_model.predict_proba(Xte_imp)
write_sub(proba_trans.argmax(1), 'svm_transductive')

# Ideas 2+1: self-trained on the test pool
self_model = self_train(Xtr_imp, y, Xte_imp, verbose=True)
proba_self = self_model.predict_proba(Xte_imp)
write_sub(proba_self.argmax(1), 'svm_selftrain')

# Ideas 2+1+3: + prior matching
write_sub((proba_self * PRIOR_W).argmax(1), 'svm_selftrain_prior')

wrote ../outputs/svm_transductive_submission.csv | dist {0: np.int64(1125), 1: np.int64(1284), 2: np.int64(1291), 3: np.int64(1300)}
  round 1: kept 2237/5000 pseudo-labels (45%)
wrote ../outputs/svm_selftrain_submission.csv | dist {0: np.int64(1130), 1: np.int64(1286), 2: np.int64(1280), 3: np.int64(1304)}
wrote ../outputs/svm_selftrain_prior_submission.csv | dist {0: np.int64(1136), 1: np.int64(1286), 2: np.int64(1272), 3: np.int64(1306)}


,id,sleep_stage
0,9000,3
1,9001,3
2,9002,1
3,9003,3
4,9004,3
...,...,...
4995,13995,0
4996,13996,1
4997,13997,2
4998,13998,1


## Takeaways
- **Transductive imputation** is the free win: more data for the one lever that works,
  zero label leakage. Use it as your new safe baseline if Step 1 held.
- **Self-training** is the swing for the fences - validated on **held-out real labels**,
  class 2 excluded so it can't amplify your weakest spot. Trust it only if Step 3 cleared
  the noise band; otherwise it's a leaderboard gamble, so A/B it against transductive.
- **Prior matching** is a cheap macro-F1 nudge; keep it only if Step 4's nested gain held.
- Anchor: keep `svm_iterimpute_submission.csv` (0.85456) as one final slot no matter what.